# AUTONOMOUS MOVIE STUDIO\n## Real Qwen GPU Validation\nThis notebook installs dependencies, clones the pinned branch, checks GPU, runs the real-Qwen pipeline, validates artifacts and downloads outputs.

In [ ]:
REPO = 'https://github.com/asdfhgds/automovies.git'\nBRANCH = 'asdfhgds-autonomous-movie-studio-spec'\nMODEL = 'Qwen/Qwen3-7B-A0.5B'\n!nvidia-smi\nimport subprocess\nif subprocess.run(['nvidia-smi']).returncode != 0: raise RuntimeError('GPU REQUIRED: Runtime → Change runtime type → GPU')

In [ ]:
!git clone --branch $BRANCH --single-branch $REPO automovies\n%cd automovies\n!bash scripts/colab_setup.sh\nimport os\nos.environ.update({'STUDIO_PROFILE':'colab-gpu','REQUIRE_REAL_LLM':'true','DIRECTOR_PROVIDER':'qwen','DIRECTOR_MODEL':MODEL,'DIRECTOR_DEVICE':'cuda','SCRIPT_PROVIDER':'qwen','SCRIPT_MODEL':MODEL,'SCRIPT_DEVICE':'cuda','CREATIVE_DIRECTOR_ENABLED':'true'})\n!python src/main.py doctor

In [ ]:
import subprocess, re, json, pathlib, time\nsubprocess.run(['python','tests/fixtures/generate_test_fixture.py','tests/fixtures/test_speech.mp4','Short legal GPU test'], check=True)\ncreated = subprocess.check_output(['python','src/main.py','init','--title','Qwen GPU Validation','--source','tests/fixtures/test_speech.mp4'], text=True)\nprint(created)\nPROJECT_ID = re.search(r'([0-9a-f-]{36})', created).group(1)\nstarted=time.time(); subprocess.run(['python','src/main.py','run','--project-id',PROJECT_ID], check=True); print('Pipeline seconds:', round(time.time()-started,2))\nproject=pathlib.Path('data')/PROJECT_ID\nrequired=['transcripts/transcript.json','scenes/scene_index.json','scenes/scene_ranking.json','scenes/selected_scenes.json','director_plan.json','script.json','timeline/timeline.json','renders/final_render.mp4','reports/qc_report.json']\nfor item in required: print(f'{item:38}', 'PASS' if (project/item).exists() else 'FAIL')\nplan=json.loads((project/'director_plan.json').read_text()); script=json.loads((project/'script.json').read_text())\nassert plan['provider_metadata']['model']==MODEL and plan['provider_metadata']['device']=='cuda'\nassert script['provider']=='qwen' and script['provider_metadata']['model']==MODEL and script['provider_metadata']['device']=='cuda'\n!ffprobe -v error -show_entries format=duration:stream=codec_name,width,height,r_frame_rate,sample_rate -of json $project/renders/final_render.mp4\nreport=pathlib.Path('GPU_VALIDATION_FINAL_REPORT.md'); report.write_text(f'# GPU Validation\n\nProject: {PROJECT_ID}\n\nReal Qwen execution: PASS\n')\nfrom google.colab import files\nfiles.download(str(project/'renders/final_render.mp4')); files.download(str(report))